# Colab training driver

Thin front-end for the package: the first cell bootstraps (mount Drive, read config, clone, install); the rest are calls into `src.colab` and the experiment drivers.

**Configure a run** by creating a config file on your Drive at `MyDrive/cross-dataset-drift-soybean-disease.env` — copy the repo's `.env.example` and fill it in: `GITHUB_PAT` (a read-only token), `GITHUB_REPO_URL`, `GIT_BRANCH`, `DRIVE_ROOT`. The bootstrap reads that file (then environment variables, then Colab Secrets), so nothing account-specific is baked into the notebook and anyone can re-run by editing their own config file.

Dataset zips live at `MyDrive/<DRIVE_ROOT>/data/raw/{ASDID,MH-SoyaHealthVision,PlantVillage}.zip`; checkpoints/results/logs persist under `MyDrive/<DRIVE_ROOT>/`. Deps install from `requirements.txt`, so this runs on whatever Python Colab provides (local dev is pinned to 3.12).

**Which experiment runs is decided by the config's `experiment:` key** (see the run cell), so swapping the `CONFIG` filename actually changes what executes.

## 1 · Setup — run every session
Mount Drive, clone the repo at your branch, install dependencies. Re-run cells 1 and 2 top-to-bottom in every new Colab session.

In [ ]:
# --- Bootstrap: mount Drive, read config, clone, install (Colab) ---
import os, shutil, subprocess, sys
from pathlib import Path

# Reduce CUDA fragmentation OOMs on the long runs (set before torch is imported anywhere).
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

from google.colab import drive
drive.mount('/content/drive')

# Edit this file on your Drive to configure a run (copy from the repo's .env.example).
CONFIG_ENV = Path('/content/drive/MyDrive/cross-dataset-drift-soybean-disease.env')
KEYS = ('GITHUB_PAT', 'GITHUB_REPO_URL', 'GIT_BRANCH', 'DRIVE_ROOT')

def _parse_env(path):
    out = {}
    if path.exists():
        for line in path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                out[k.strip()] = v.strip()
    return out

# 1) Drive config file, 2) environment, 3) Colab Secrets (last; unavailable outside the Colab UI)
SECRETS = _parse_env(CONFIG_ENV)
for k in KEYS:
    if os.environ.get(k):
        SECRETS[k] = os.environ[k]
if not SECRETS.get('GITHUB_PAT'):
    try:    
        from google.colab import userdata
        SECRETS['GITHUB_PAT'] = userdata.get('GITHUB_PAT')
    except Exception as e:
        print('Colab Secrets unavailable:', type(e).__name__)

missing = [k for k in ('GITHUB_PAT', 'GITHUB_REPO_URL', 'DRIVE_ROOT') if not SECRETS.get(k)]
if missing:
    raise RuntimeError(f'Missing {missing}. Create {CONFIG_ENV} from the repo .env.example (or set them as env vars).')

PAT        = SECRETS['GITHUB_PAT']
REPO       = SECRETS['GITHUB_REPO_URL']
BRANCH     = SECRETS.get('GIT_BRANCH', 'main')
DRIVE_ROOT = SECRETS['DRIVE_ROOT']

CODE = Path('/content/code')
if CODE.exists():
    shutil.rmtree(CODE)
url = REPO.replace('https://', f'https://{PAT}@')
subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH, url, str(CODE)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements.txt')], check=True)
sys.path.insert(0, str(CODE))
print('bootstrap done:', CODE)

In [ ]:
# --- Sanity check: confirm the clone is the version you expect ---
import subprocess
print(subprocess.run(['git', '-C', str(CODE), 'log', '-1', '--oneline'],
                     capture_output=True, text=True).stdout.strip())

from src.config import load_config
from src.experiments._common import seed_pairs
cfg = load_config()
print('seed_design:', cfg.seed_design)
print('pairs:', seed_pairs(cfg))

## 2 · Data & paths — run every session
Cache datasets to local SSD and resolve `paths` (data on SSD, checkpoints/results on Drive). `paths` is consumed by every stage below.

In [ ]:
# --- Cache datasets to local SSD, pre-resize once, target artifacts at Drive ---
import src.colab as colab

DRIVE = Path('/content/drive/MyDrive') / DRIVE_ROOT
LOCAL = Path('/content/local')

local_raw = colab.cache_datasets(DRIVE / 'data' / 'raw', LOCAL)
colab.pre_resize_images(local_raw)
paths = colab.resolve_paths(LOCAL, DRIVE)
print('data:', paths.data_root, '| artifacts:', paths.checkpoints_dir)

## 3 · Run a stage: training **or** evaluation
Dispatch is by the config's `experiment:` key — change `CONFIG` to pick what runs:

- **Training** (writes checkpoints): `full_finetune.yaml`, `unweighted_mh.yaml`, `label_smoothing.yaml`, `control_study.yaml`, `linear_probe.yaml`
- **Evaluation** (reads checkpoints → `eval_results.csv`): `evaluate.yaml` / `evaluate_resume.yaml`

Run the training configs first (GPU), then evaluation. **Skip this stage** if the checkpoints and `eval_results.csv` already exist on Drive.

In [ ]:
# --- Run an experiment (training only; evaluation runs locally) ---
# Dispatch is by the config's `experiment:` key, so changing CONFIG changes what runs.
#   full_finetune.yaml     -> baseline 16-model space      (checkpoints/finetune)
#   unweighted_mh.yaml     -> unweighted-loss ablation     (checkpoints/finetune_unweighted)
#   label_smoothing.yaml   -> label-smoothing variant      (checkpoints/finetune_label_smoothing)
#   control_study.yaml     -> matched-ASDID control study  (checkpoints/control_study)
#   linear_probe.yaml      -> frozen-feature probes        (checkpoints/linear_probe)
#   calibration.yaml       -> quick single-seed timing pass
import importlib, logging
import yaml
logging.basicConfig(level=logging.INFO, format='%(message)s')
from src.config import load_config

EXPERIMENTS = {
    'finetune': 'src.experiments.finetune',
    'control_study': 'src.experiments.robustness.control_study',
    'linear_probe': 'src.experiments.robustness.linear_probe',
    'evaluate': 'src.experiments.robustness.evaluate',
}

CONFIG = CODE / 'configs' / 'experiments' / 'evaluate_resume.yaml'   # <-- change this per run

experiment = (yaml.safe_load(CONFIG.read_text()) or {}).get('experiment')
assert experiment in EXPERIMENTS, f"{CONFIG.name} must set experiment: to one of {sorted(EXPERIMENTS)}"
cfg = load_config(str(CONFIG), paths=paths)
# run_name only governs the finetune driver's checkpoint namespace; the
# control_study / linear_probe / evaluate drivers use their own fixed namespace.
tag = f'  (run_name={cfg.run_name})' if experiment == 'finetune' else ''
print(f'running {experiment}{tag}  from {CONFIG.name}')
importlib.import_module(EXPERIMENTS[experiment]).run(cfg)

## 4 · Standard analysis — needs only `eval_results.csv`
Aggregates, figures, the significance tests, and the decomposition ordering-robustness check. These read CSVs only, so they run without a GPU.

In [ ]:
import sys, runpy
for script in ("aggregate_results", "make_figures", "compute_significance", "compute_decomposition_robustness"):
    sys.argv = [script, "--results-dir", f"{DRIVE}/results"]
    runpy.run_path(str(CODE / "scripts" / f"{script}.py"), run_name="__main__")

## 5 · Checkpoint-based analysis — needs the trained checkpoints
ECE bin-count robustness, class-vs-dataset feature silhouettes (UMAP), and the few-shot adaptation curve. Run after training + evaluation. **No retraining** — they load existing checkpoints and only do inference + analysis. Feature geometry needs `umap-learn`.

In [ ]:
# Checkpoint-based analysis: loads existing checkpoints (no retraining).
# Default = full 3x3 split x init seed grid (matches the committed result CSVs).
# Pass quick=True for the faster thesis subset (fixed split seed x init seeds).
!pip install -q umap-learn
from src.config import load_config
from scripts.compute_ece_robustness import run as run_ece
from scripts.compute_feature_geometry import run as run_fg
from scripts.compute_few_shot import run as run_fs
from scripts.compute_coral import run as run_coral

cfg = load_config(paths=paths)
#run_ece(cfg)               # -> results/ece_robustness.csv     (quick=True for thesis subset)
#run_fg(cfg, per_class=25)  # -> results/feature_geometry.csv   (quick=True for thesis subset)
#run_fs(cfg)                # -> results/few_shot.csv           (quick=True for thesis subset)
run_coral(cfg)             # -> results/coral.csv              (zero-label DA baseline)

## 6 · Bundle artifacts to Drive

Checkpoints already persist under `MyDrive/<DRIVE_ROOT>/checkpoints/`. This zips them (plus any logs/results) for easy download; evaluation is then run locally against these checkpoints.

In [ ]:
colab.bundle_artifacts(paths, DRIVE)